# Module 18 Lab — OpenAI Agents SDK: Triage, Handoffs, and Guardrails

In this lab you will build a customer support system that demonstrates two safety mechanisms that every production agent should have:

```
User message
    ↓ [InputGuardrail: PII check]
Triage Agent
    ├── handoff → Billing Specialist (has lookup_invoice)
    └── handoff → Tech Support Specialist (has run_diagnostic)
    ↓ [OutputGuardrail: LLM-as-judge safety check]
Final response
```

**By the end of this lab you will be able to:**
- Define specialist agents with `Agent` and connect them via `handoff`
- Write an `InputGuardrail` that blocks PII before the agent sees it
- Write an `OutputGuardrail` that uses a second LLM to judge response safety
- Catch `InputGuardrailTripwireTriggered` and return a user-friendly message
- Control execution with `RunConfig`

**Prerequisites:** OpenAI API key. Get one at https://platform.openai.com/api-keys

**Time estimate:** 45–60 minutes

## Setup — Install Dependencies

In [ ]:
!pip install openai-agents pydantic

## Environment Setup

In Google Colab, add your API key as a secret named `OPENAI_API_KEY` (key icon in the left sidebar), then run the cell below.

Locally: `export OPENAI_API_KEY=your_key_here`

In [ ]:
import os
import asyncio
import re
from pydantic import BaseModel
from agents import (
    Agent, Runner, function_tool, handoff,
    InputGuardrail, OutputGuardrail, GuardrailFunctionOutput,
    RunContextWrapper, RunConfig,
)
from agents.exceptions import InputGuardrailTripwireTriggered

# In Colab: load from Colab secrets
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass  # not in Colab — rely on environment variable

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    raise ValueError("Set OPENAI_API_KEY in environment")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
DEFAULT_MODEL = "gpt-4o-mini"
print(f"Using model: {DEFAULT_MODEL}")
print("Environment ready.")

## Section 1 — Tools

In the OpenAI Agents SDK, `@function_tool` works similarly to ADK's `@tool`:
- The **docstring** becomes the tool description the model reads
- **Type hints** generate the JSON schema
- Functions must be synchronous or async — the SDK handles both

These two tools are stubs that return canned data. In production they would query a billing database and a monitoring API.

In [ ]:
@function_tool
def lookup_invoice(invoice_id: str) -> str:
    """Look up an invoice by ID and return its details."""
    invoices = {
        "INV-001": "Amount: $99.00, Status: Paid",
        "INV-002": "Amount: $149.00, Status: Pending",
    }
    return invoices.get(invoice_id, f"Invoice {invoice_id} not found")


@function_tool
def run_diagnostic(component: str) -> str:
    """Run a diagnostic check on a system component."""
    return f"Diagnostic for {component}: All systems operational. Last checked: 2025-01-01."


print("Tools defined: lookup_invoice, run_diagnostic")

## Section 2 — Specialist Agents

Each specialist `Agent` handles one domain. The triage agent will later route to them via `handoff`. Specialists should have:
- A focused `instructions` prompt that defines their role and limits
- Only the tools relevant to their domain — do not give billing tools to tech support

### TODO 1 — Billing Specialist

In [ ]:
# TODO 1 — Create the billing specialist agent
# It should have access to lookup_invoice and a clear instruction
# about what it can and cannot do (e.g. it cannot issue refunds directly).

billing_agent = Agent(
    name="Billing Specialist",
    instructions=None,  # TODO: write a clear billing support instruction
    tools=[lookup_invoice],
    model=DEFAULT_MODEL,
)

# === SOLUTION (reveal only if stuck) ===
# billing_agent = Agent(
#     name="Billing Specialist",
#     instructions=(
#         "You are a billing support specialist. "
#         "Use lookup_invoice to retrieve invoice details when the customer provides an invoice ID. "
#         "You can explain invoice statuses and amounts. "
#         "You CANNOT issue refunds, change prices, or make billing adjustments — "
#         "direct the customer to billing@company.com for those requests."
#     ),
#     tools=[lookup_invoice],
#     model=DEFAULT_MODEL,
# )

print("billing_agent:", billing_agent.name)

### TODO 2 — Tech Support Specialist

In [ ]:
# TODO 2 — Create the tech support specialist agent
# It should have access to run_diagnostic and appropriate instructions.

tech_agent = Agent(
    name="Tech Support Specialist",
    instructions=None,  # TODO: write a clear tech support instruction
    tools=[run_diagnostic],
    model=DEFAULT_MODEL,
)

# === SOLUTION (reveal only if stuck) ===
# tech_agent = Agent(
#     name="Tech Support Specialist",
#     instructions=(
#         "You are a technical support specialist. "
#         "Use run_diagnostic to check the status of system components the customer mentions. "
#         "Provide clear, step-by-step troubleshooting guidance. "
#         "If a component is down, escalate to the engineering team — do not promise a fix time."
#     ),
#     tools=[run_diagnostic],
#     model=DEFAULT_MODEL,
# )

print("tech_agent:", tech_agent.name)

## Section 3 — PII Input Guardrail

An `InputGuardrail` runs **before** the agent processes the user message. If it sets `tripwire_triggered=True`, the SDK raises `InputGuardrailTripwireTriggered` and the agent never sees the message.

The guardrail function signature is:
```python
async def my_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str,
) -> GuardrailFunctionOutput:
    ...
```

`GuardrailFunctionOutput` takes:
- `output_info` — any dict you want to attach for logging (not shown to the user)
- `tripwire_triggered` — `True` blocks the message, `False` lets it through

### TODO 3 — PII Guardrail

Check for three PII patterns using `re.search`:
- SSN: `\d{3}-\d{2}-\d{4}`
- Credit card: `\d{4}[\s-]\d{4}[\s-]\d{4}[\s-]\d{4}`
- Date of birth mention: `(born|dob|date of birth)` (case-insensitive)

In [ ]:
# TODO 3 — Implement the PII guardrail

async def pii_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str,
) -> GuardrailFunctionOutput:
    """Block messages that contain PII before the agent sees them."""
    # TODO: check for SSN, credit card, and DOB patterns
    # Return GuardrailFunctionOutput with appropriate tripwire_triggered value
    pii_detected = False  # TODO: replace with real detection logic
    return GuardrailFunctionOutput(
        output_info={"pii_detected": pii_detected},
        tripwire_triggered=pii_detected,
    )

# === SOLUTION (reveal only if stuck) ===
# SSN_PATTERN = re.compile(r"\d{3}-\d{2}-\d{4}")
# CC_PATTERN = re.compile(r"\d{4}[\s-]\d{4}[\s-]\d{4}[\s-]\d{4}")
# DOB_PATTERN = re.compile(r"(born|dob|date of birth)", re.IGNORECASE)
#
# async def pii_guardrail(
#     ctx: RunContextWrapper,
#     agent: Agent,
#     input: str,
# ) -> GuardrailFunctionOutput:
#     """Block messages that contain PII before the agent sees them."""
#     pii_detected = bool(
#         SSN_PATTERN.search(input)
#         or CC_PATTERN.search(input)
#         or DOB_PATTERN.search(input)
#     )
#     return GuardrailFunctionOutput(
#         output_info={"pii_detected": pii_detected},
#         tripwire_triggered=pii_detected,
#     )

print("pii_guardrail defined")

## Section 4 — Output Guardrail (LLM-as-Judge)

An `OutputGuardrail` runs **after** the agent produces a response, before it is returned to the user. It has the same `tripwire_triggered` mechanism.

The pattern below uses a dedicated `safety_checker` agent as the judge. This is the **LLM-as-judge** pattern: a second, cheaper model call that inspects the first model's output for policy violations.

Using `output_type=SafetyCheck` forces the judge to return structured JSON matching the Pydantic model — no regex parsing needed.

This cell is given — read it carefully before continuing.

In [ ]:
class SafetyCheck(BaseModel):
    is_safe: bool
    reason: str


safety_checker = Agent(
    name="Safety Checker",
    instructions=(
        "You check if customer support responses make promises the company cannot keep. "
        "Examples of unsafe promises: guaranteed refunds, free upgrades, impossible deadlines, "
        "commitments to fix things by a specific time. "
        "Return is_safe=False if any such promises are detected. "
        "Return is_safe=True if the response is appropriately cautious."
    ),
    output_type=SafetyCheck,
    model=DEFAULT_MODEL,
)


async def output_safety_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    output: str,
) -> GuardrailFunctionOutput:
    """Use a safety checker LLM to judge whether the response makes unsafe promises."""
    result = await Runner.run(safety_checker, output, context=ctx.context)
    check: SafetyCheck = result.final_output
    return GuardrailFunctionOutput(
        output_info={"reason": check.reason, "is_safe": check.is_safe},
        tripwire_triggered=not check.is_safe,
    )


print("output_safety_guardrail defined")

## Section 5 — Triage Agent

The triage agent is the entry point. It has:
- **No tools of its own** — it delegates all real work via handoffs
- **`handoffs`** — a list of `handoff(agent)` calls that tell the SDK which agents it can transfer to
- **`input_guardrails`** — list of `InputGuardrail` objects wrapping your guardrail functions
- **`output_guardrails`** — list of `OutputGuardrail` objects wrapping your guardrail functions

When the triage agent decides to hand off, the SDK runs the specialist agent and returns its response as the final output (after passing it through any output guardrails).

### TODO 4 — Triage Agent

In [ ]:
# TODO 4 — Create the triage agent with both guardrails and both handoffs

triage_agent = Agent(
    name="Triage",
    instructions=None,  # TODO: write routing instructions
    handoffs=[],        # TODO: add handoff(billing_agent) and handoff(tech_agent)
    input_guardrails=[], # TODO: wrap pii_guardrail in InputGuardrail
    output_guardrails=[], # TODO: wrap output_safety_guardrail in OutputGuardrail
    model=DEFAULT_MODEL,
)

# === SOLUTION (reveal only if stuck) ===
# triage_agent = Agent(
#     name="Triage",
#     instructions=(
#         "You are a customer support triage agent. "
#         "Route billing questions (invoices, payments, charges) to the Billing Specialist. "
#         "Route technical questions (connectivity, diagnostics, system issues) to the Tech Support Specialist. "
#         "If a question spans both, route to the most relevant specialist first."
#     ),
#     handoffs=[handoff(billing_agent), handoff(tech_agent)],
#     input_guardrails=[InputGuardrail(guardrail_function=pii_guardrail)],
#     output_guardrails=[OutputGuardrail(guardrail_function=output_safety_guardrail)],
#     model=DEFAULT_MODEL,
# )

print("triage_agent:", triage_agent.name)

## Section 6 — Runner with Error Handling

The `Runner.run` call is async and returns a `RunResult`. The key fields are:
- `result.final_output` — the last agent's response as a string
- `result.last_agent` — which agent produced the final response (useful for debugging handoffs)

`RunConfig` controls execution parameters. `max_turns` limits the total number of agent turns (including handoffs and tool calls) to prevent runaway loops.

When an input guardrail trips, `Runner.run` raises `InputGuardrailTripwireTriggered` before any agent call is made. Always catch this and return a helpful message — never leak the raw exception to the user.

### TODO 5 — `handle_query` Function

In [ ]:
# TODO 5 — Implement handle_query
# Call Runner.run with triage_agent, the message, and RunConfig(max_turns=8).
# Catch InputGuardrailTripwireTriggered and return a user-friendly refusal.
# Print result.final_output on success.

async def handle_query(message: str) -> str:
    """Route a customer query through the triage system and return the response."""
    # TODO: implement this function
    return "Not implemented yet"

# === SOLUTION (reveal only if stuck) ===
# async def handle_query(message: str) -> str:
#     """Route a customer query through the triage system and return the response."""
#     try:
#         result = await Runner.run(
#             triage_agent,
#             message,
#             run_config=RunConfig(max_turns=8),
#         )
#         response = result.final_output
#         print(f"[Handled by: {result.last_agent.name}]")
#         print(response)
#         return response
#     except InputGuardrailTripwireTriggered:
#         msg = (
#             "I'm sorry, but I cannot process messages containing personal information "
#             "such as Social Security numbers, credit card numbers, or dates of birth. "
#             "Please remove that information and try again."
#         )
#         print(msg)
#         return msg

print("handle_query defined")

## Section 7 — Test Cases

Run all three test cases. Verify that:
1. The billing query is handled by the Billing Specialist
2. The connectivity query is handled by the Tech Support Specialist
3. The SSN query is blocked by the PII guardrail before reaching any agent

The third case should print your user-friendly refusal message, not a traceback.

In [ ]:
test_cases = [
    "I need help with invoice INV-001",
    "My connection keeps dropping — can you run a diagnostic?",
    "My SSN is 123-45-6789 and I need a refund",  # triggers PII guardrail
]

for query in test_cases:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    asyncio.run(handle_query(query))

## Section 8 — Exploration Exercises

Try at least two of the following to deepen your understanding:

### Exercise A — Add a third specialist
Create a `returns_agent` that handles product returns and refund eligibility checks. Add it to the triage agent's handoffs and update the triage instructions.

### Exercise B — Test the output guardrail
Modify the billing or tech agent's instructions to make it say something like "I guarantee we'll fix this within 24 hours" — then run a test query and verify the output guardrail trips. What does the user see?

### Exercise C — Inspect handoff metadata
Print `result.new_items` after a successful run. Each item in the list is an event — tool calls, handoff decisions, model responses. Find the handoff event and inspect its structure.

### Exercise D — Multi-turn conversation
The `Runner.run` API supports conversation history via the `input` parameter accepting a list of messages. Implement a simple REPL that passes previous turns forward so the specialist can reference earlier context.

In [ ]:
# Your exploration code here


## Reflection Questions

Answer these before finishing the lab:

1. **Guardrail order:** Input guardrails run before the agent, output guardrails after. Could you implement PII detection as an output guardrail instead? What would be the trade-off?

2. **LLM-as-judge cost:** The output safety guardrail makes an extra LLM call for every response. In a high-volume system, how would you reduce this cost without removing the guardrail entirely?

3. **Handoff vs tool:** The triage agent uses `handoff` to transfer to specialists. You could instead give the triage agent the specialist's tools directly. What are the advantages of the handoff pattern over this alternative?

4. **False positives:** The PII regex for credit cards (`\d{4}[\s-]\d{4}[\s-]\d{4}[\s-]\d{4}`) would also block a message like "Check order 1234-5678-9012-3456". How would you reduce false positives while keeping real PII protection?

In [ ]:
# Your reflection notes

reflections = """
1. PII as output guardrail:

2. Reducing LLM-as-judge cost:

3. Handoff vs tool:

4. Reducing false positives:
"""

print(reflections)

## Checkpoint

Before moving on, verify you can:

- [ ] Create specialist agents with `Agent` and connect them via `handoff`
- [ ] Write an `InputGuardrail` that detects PII using regex and trips the tripwire
- [ ] Explain how the LLM-as-judge pattern works and why `output_type=SafetyCheck` is useful
- [ ] Catch `InputGuardrailTripwireTriggered` and return a user-friendly message
- [ ] Identify which agent handled a request using `result.last_agent`
- [ ] Explain when to use `handoff` vs giving tools directly to the triage agent